# 01 — Data Understanding dan Exploratory Data Analysis (EDA)

**Project:** NutriSmart AI  
**Dataset utama:** `ObesityDataSet_raw_and_data_sinthetic.csv`

## Tujuan notebook

Notebook ini digunakan untuk:

1. Memahami sumber, ukuran, dan struktur dataset.
2. Memahami arti setiap variabel.
3. Memeriksa kualitas data: missing value, duplikat, tipe data, dan rentang nilai.
4. Mengamati distribusi fitur numerik dan kategorikal.
5. Mengamati distribusi target `NObeyesdad`.
6. Mengidentifikasi hubungan awal antara pola hidup dan tingkat obesitas.
7. Menentukan keputusan awal sebelum tahap preprocessing dan modeling.

> **Batas notebook:** Notebook ini belum melakukan encoding, pembagian train-test, training model, atau pemilihan model. Semua proses tersebut dilakukan pada notebook berikutnya.


## 1. Informasi sumber dataset

Dataset berasal dari **UCI Machine Learning Repository** dengan judul:

**Estimation of Obesity Levels Based on Eating Habits and Physical Condition**

Dataset memuat data individu dari Meksiko, Peru, dan Kolombia. Dataset terdiri dari 2.111 observasi dan 17 kolom. Menurut dokumentasi sumber, sekitar 23% data dikumpulkan melalui platform web dan sekitar 77% data dibentuk secara sintetik menggunakan Weka dan SMOTE.

Sumber:

- [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition)
- [Artikel pengantar dataset](https://doi.org/10.1016/j.dib.2019.104344)

### Implikasi bagi project

- Dataset sesuai untuk eksperimen klasifikasi.
- Dataset tidak secara khusus merepresentasikan populasi Indonesia.
- Sebagian besar data bersifat sintetik sehingga performa model harus ditafsirkan secara hati-hati.
- Hasil aplikasi hanya digunakan sebagai **skrining berbasis data**, bukan diagnosis medis.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

print("Library berhasil dimuat.")


## 2. Menentukan lokasi project dan membaca dataset

Kode berikut dibuat agar notebook tetap dapat dijalankan ketika Jupyter dibuka dari:

- folder utama project; atau
- folder `notebooks/`.


In [ ]:
DATASET_FILENAME = "ObesityDataSet_raw_and_data_sinthetic.csv"

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / DATASET_FILENAME).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan file berada di "
        "data/raw/ObesityDataSet_raw_and_data_sinthetic.csv"
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / DATASET_FILENAME

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset path : {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print("Dataset berhasil dibaca.")


In [ ]:
display(df.head())


## 3. Data dictionary

Kelompok variabel pada dataset:

- **Karakteristik individu:** `Gender`, `Age`, `Height`, `Weight`, dan riwayat keluarga.
- **Pola makan:** `FAVC`, `FCVC`, `NCP`, `CAEC`, `CH2O`, dan `CALC`.
- **Aktivitas dan perilaku:** `SCC`, `FAF`, `TUE`, `SMOKE`, dan `MTRANS`.
- **Target:** `NObeyesdad`.

Sebagian fitur seperti `FCVC`, `CH2O`, `FAF`, dan `TUE` memiliki rentang skor tertentu. Karena sebagian besar data dibuat secara sintetik, dataset juga mengandung nilai desimal di antara skor kategorinya.


In [ ]:
data_dictionary = pd.DataFrame(
    [
        ("Gender", "Kategorikal", "Jenis kelamin", "Male, Female"),
        ("Age", "Numerik", "Usia individu dalam tahun", "14–61 pada dataset"),
        ("Height", "Numerik", "Tinggi badan dalam meter", "Nilai kontinu"),
        ("Weight", "Numerik", "Berat badan dalam kilogram", "Nilai kontinu"),
        (
            "family_history_with_overweight",
            "Kategorikal",
            "Riwayat keluarga dengan kelebihan berat badan",
            "yes, no",
        ),
        (
            "FAVC",
            "Kategorikal",
            "Sering mengonsumsi makanan tinggi kalori",
            "yes, no",
        ),
        (
            "FCVC",
            "Numerik/ordinal",
            "Frekuensi konsumsi sayuran",
            "Skala 1–3",
        ),
        (
            "NCP",
            "Numerik/ordinal",
            "Jumlah makanan utama",
            "Skala 1–4",
        ),
        (
            "CAEC",
            "Kategorikal ordinal",
            "Konsumsi makanan di antara waktu makan",
            "no, Sometimes, Frequently, Always",
        ),
        ("SMOKE", "Kategorikal", "Kebiasaan merokok", "yes, no"),
        (
            "CH2O",
            "Numerik/ordinal",
            "Konsumsi air harian",
            "Skala 1–3",
        ),
        (
            "SCC",
            "Kategorikal",
            "Kebiasaan memantau konsumsi kalori",
            "yes, no",
        ),
        (
            "FAF",
            "Numerik/ordinal",
            "Frekuensi aktivitas fisik",
            "Skala 0–3",
        ),
        (
            "TUE",
            "Numerik/ordinal",
            "Waktu menggunakan perangkat teknologi",
            "Skala 0–2",
        ),
        (
            "CALC",
            "Kategorikal ordinal",
            "Konsumsi alkohol",
            "no, Sometimes, Frequently, Always",
        ),
        (
            "MTRANS",
            "Kategorikal",
            "Moda transportasi utama",
            "Automobile, Bike, Motorbike, Public Transportation, Walking",
        ),
        (
            "NObeyesdad",
            "Kategorikal target",
            "Tingkat/kategori obesitas",
            "7 kelas",
        ),
    ],
    columns=["Kolom", "Tipe Konseptual", "Deskripsi", "Nilai/Rentang"],
)

display(data_dictionary)


## 4. Struktur dasar dataset

Bagian ini memeriksa:

- jumlah baris dan kolom;
- nama kolom;
- tipe data;
- penggunaan memori;
- jumlah nilai unik.


In [ ]:
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")


In [ ]:
print("Daftar kolom:")
for index, column in enumerate(df.columns, start=1):
    print(f"{index:>2}. {column}")


In [ ]:
df.info()


In [ ]:
column_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
    }
).sort_values(["dtype", "unique_values"])

display(column_summary)


### Interpretasi awal tipe data

Kolom `Age`, `FCVC`, `NCP`, `CH2O`, `FAF`, dan `TUE` disimpan sebagai `float64`. Nilai tersebut tidak selalu bulat karena sebagian data dibentuk secara sintetik.

Pada tahap preprocessing berikutnya, kita tidak boleh langsung membulatkan nilai tanpa alasan. Nilai dapat dipertahankan sebagai numerik selama sesuai dengan rentang dokumentasinya.


In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()

print("Kolom numerik:")
print(numeric_columns)

print("\nKolom kategorikal:")
print(categorical_columns)


## 5. Pemeriksaan kualitas data

Pemeriksaan meliputi:

1. Missing value.
2. Duplikat.
3. Nilai di luar rentang yang terdokumentasi.
4. Konsistensi nilai kategorikal.


In [ ]:
missing_summary = (
    pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_percentage": (df.isna().mean() * 100).round(2),
        }
    )
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)
print(f"Total missing value: {int(df.isna().sum().sum())}")


In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_percentage = duplicate_count / len(df) * 100

print(f"Jumlah baris duplikat penuh : {duplicate_count}")
print(f"Persentase duplikat         : {duplicate_percentage:.2f}%")

if duplicate_count > 0:
    print("\nContoh baris yang memiliki duplikat:")
    duplicate_rows = df[df.duplicated(keep=False)].sort_values(
        by=list(df.columns)
    )
    display(duplicate_rows.head(10))


### Keputusan awal terkait duplikat

Baris duplikat penuh sebaiknya dihapus pada notebook preprocessing **sebelum** pembagian data training dan testing. Hal ini mengurangi risiko observasi identik muncul pada kedua bagian data dan membuat performa model terlihat terlalu tinggi.

Dataset mentah pada folder `data/raw/` tidak diubah pada notebook ini.


In [ ]:
documented_ranges = {
    "FCVC": (1, 3),
    "NCP": (1, 4),
    "CH2O": (1, 3),
    "FAF": (0, 3),
    "TUE": (0, 2),
}

range_checks = []

for column, (minimum, maximum) in documented_ranges.items():
    invalid_count = int((~df[column].between(minimum, maximum)).sum())
    range_checks.append(
        {
            "column": column,
            "documented_min": minimum,
            "observed_min": df[column].min(),
            "documented_max": maximum,
            "observed_max": df[column].max(),
            "invalid_count": invalid_count,
        }
    )

range_check_df = pd.DataFrame(range_checks)
display(range_check_df)


In [ ]:
allowed_categories = {
    "Gender": {"Male", "Female"},
    "family_history_with_overweight": {"yes", "no"},
    "FAVC": {"yes", "no"},
    "CAEC": {"no", "Sometimes", "Frequently", "Always"},
    "SMOKE": {"yes", "no"},
    "SCC": {"yes", "no"},
    "CALC": {"no", "Sometimes", "Frequently", "Always"},
    "MTRANS": {
        "Automobile",
        "Bike",
        "Motorbike",
        "Public_Transportation",
        "Walking",
    },
}

category_checks = []

for column, allowed_values in allowed_categories.items():
    observed_values = set(df[column].dropna().unique())
    unexpected_values = sorted(observed_values - allowed_values)

    category_checks.append(
        {
            "column": column,
            "observed_unique": len(observed_values),
            "unexpected_values": unexpected_values,
            "is_valid": len(unexpected_values) == 0,
        }
    )

display(pd.DataFrame(category_checks))


## 6. Statistik deskriptif fitur numerik

Statistik ini membantu memahami:

- nilai minimum dan maksimum;
- rata-rata dan median;
- penyebaran data;
- kemungkinan nilai ekstrem;
- karakteristik skala setiap variabel.


In [ ]:
numeric_summary = df[numeric_columns].describe().T
numeric_summary["median"] = df[numeric_columns].median()
numeric_summary["skewness"] = df[numeric_columns].skew()
numeric_summary = numeric_summary[
    ["count", "mean", "std", "min", "25%", "median", "75%", "max", "skewness"]
]

display(numeric_summary)


In [ ]:
decimal_value_summary = []

for column in numeric_columns:
    decimal_percentage = ((df[column] % 1) != 0).mean() * 100
    decimal_value_summary.append(
        {
            "column": column,
            "percentage_non_integer": round(decimal_percentage, 2),
        }
    )

display(pd.DataFrame(decimal_value_summary))


### Catatan mengenai nilai desimal

Nilai desimal pada variabel yang secara konseptual berupa skala ordinal merupakan salah satu karakteristik data sintetik. Pada tahap awal, nilai tersebut akan tetap diperlakukan sebagai numerik.

Kita akan mengevaluasi kembali keputusan tersebut pada notebook preprocessing agar bentuk input aplikasi tetap mudah dipahami pengguna.


## 7. Distribusi fitur numerik

Setiap grafik dibuat terpisah agar mudah dibaca. Grafik histogram digunakan untuk melihat bentuk distribusi, sedangkan boxplot membantu melihat penyebaran dan nilai ekstrem.


In [ ]:
for column in numeric_columns:
    plt.figure(figsize=(8, 4))
    plt.hist(df[column], bins=30)
    plt.title(f"Distribusi {column}")
    plt.xlabel(column)
    plt.ylabel("Frekuensi")
    plt.tight_layout()
    plt.show()


In [ ]:
for column in numeric_columns:
    plt.figure(figsize=(8, 2.8))
    plt.boxplot(df[column].dropna(), vert=False)
    plt.title(f"Boxplot {column}")
    plt.xlabel(column)
    plt.tight_layout()
    plt.show()


## 8. Distribusi fitur kategorikal

Bagian ini menampilkan jumlah dan proporsi setiap kategori.


In [ ]:
categorical_feature_columns = [
    column for column in categorical_columns if column != "NObeyesdad"
]

for column in categorical_feature_columns:
    counts = df[column].value_counts(dropna=False)
    percentages = (df[column].value_counts(normalize=True, dropna=False) * 100).round(2)

    summary = pd.DataFrame(
        {
            "count": counts,
            "percentage": percentages,
        }
    )

    print(f"\n=== {column} ===")
    display(summary)

    plt.figure(figsize=(9, 4))
    counts.plot(kind="bar")
    plt.title(f"Distribusi {column}")
    plt.xlabel(column)
    plt.ylabel("Jumlah")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


### Hal yang perlu diperhatikan

Beberapa kategori memiliki jumlah sangat kecil, misalnya:

- pengguna yang merokok;
- pengguna yang memantau kalori;
- transportasi `Bike` dan `Motorbike`;
- konsumsi alkohol `Always`.

Ketidakseimbangan kategori fitur perlu diperhatikan karena kategori yang sangat jarang dapat sulit dipelajari oleh model.


## 9. Distribusi target asli: `NObeyesdad`

Target asli memiliki tujuh kelas:

1. `Insufficient_Weight`
2. `Normal_Weight`
3. `Overweight_Level_I`
4. `Overweight_Level_II`
5. `Obesity_Type_I`
6. `Obesity_Type_II`
7. `Obesity_Type_III`


In [ ]:
target_order = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

target_counts = df["NObeyesdad"].value_counts().reindex(target_order)
target_percentages = (
    df["NObeyesdad"].value_counts(normalize=True).reindex(target_order) * 100
).round(2)

target_summary = pd.DataFrame(
    {
        "count": target_counts,
        "percentage": target_percentages,
    }
)

display(target_summary)

plt.figure(figsize=(11, 5))
target_counts.plot(kind="bar")
plt.title("Distribusi Target NObeyesdad")
plt.xlabel("Kategori")
plt.ylabel("Jumlah")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
largest_class = target_counts.max()
smallest_class = target_counts.min()
imbalance_ratio = largest_class / smallest_class

print(f"Kelas terbesar : {target_counts.idxmax()} ({largest_class})")
print(f"Kelas terkecil : {target_counts.idxmin()} ({smallest_class})")
print(f"Rasio kelas terbesar terhadap terkecil: {imbalance_ratio:.2f}")


### Interpretasi distribusi target

Tujuh kelas target relatif seimbang. Namun, hal tersebut tidak berarti kita boleh melewatkan evaluasi per kelas.

Pada project NutriSmart AI, target nantinya direncanakan menjadi klasifikasi biner:

- **Non-Obesity**
- **Obesity**

Pembuatan target biner akan dilakukan secara formal pada notebook preprocessing.


## 10. Perhitungan BMI hanya untuk kepentingan EDA

BMI dihitung untuk memahami hubungan antara `Height`, `Weight`, dan label target.

> **Penting:** BMI, tinggi, dan berat badan tidak direncanakan sebagai fitur utama model risiko pola hidup. Jika dimasukkan, model dapat menebak label terutama dari kondisi tubuh, bukan dari pola makan dan aktivitas.


In [ ]:
df_eda = df.copy()
df_eda["BMI_EDA"] = df_eda["Weight"] / (df_eda["Height"] ** 2)

display(df_eda["BMI_EDA"].describe().to_frame("BMI_EDA"))


In [ ]:
bmi_by_target = (
    df_eda.groupby("NObeyesdad", observed=False)["BMI_EDA"]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(target_order)
)

display(bmi_by_target)


In [ ]:
bmi_group_data = [
    df_eda.loc[df_eda["NObeyesdad"] == category, "BMI_EDA"].dropna()
    for category in target_order
]

plt.figure(figsize=(12, 5))
plt.boxplot(bmi_group_data, labels=target_order, vert=True)
plt.title("Distribusi BMI Berdasarkan Target NObeyesdad")
plt.xlabel("Kategori target")
plt.ylabel("BMI hasil perhitungan")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


### Kesimpulan penting mengenai potensi leakage

Perbedaan BMI antar-kelas sangat jelas karena target memang berhubungan langsung dengan tinggi dan berat badan. Apabila `Height`, `Weight`, atau BMI digunakan sebagai prediktor, performa model dapat menjadi sangat tinggi tetapi tidak lagi menjawab tujuan utama project.

Untuk menjaga konsep:

- **BMI:** menjelaskan status tubuh saat ini.
- **Model AI:** menilai pola hidup berdasarkan pola makan dan aktivitas.


## 11. Hubungan fitur numerik dengan target asli

Tabel berikut menampilkan rata-rata fitur numerik pada setiap kategori target.


In [ ]:
numeric_by_target = (
    df_eda.groupby("NObeyesdad", observed=False)[numeric_columns + ["BMI_EDA"]]
    .mean()
    .reindex(target_order)
)

display(numeric_by_target)


In [ ]:
lifestyle_numeric_columns = ["FCVC", "NCP", "CH2O", "FAF", "TUE"]

for column in lifestyle_numeric_columns:
    group_data = [
        df_eda.loc[df_eda["NObeyesdad"] == category, column].dropna()
        for category in target_order
    ]

    plt.figure(figsize=(12, 5))
    plt.boxplot(group_data, labels=target_order, vert=True)
    plt.title(f"Distribusi {column} Berdasarkan Target")
    plt.xlabel("Kategori target")
    plt.ylabel(column)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 12. Hubungan fitur kategorikal dengan target asli

Crosstab dinormalisasi per kategori fitur. Setiap baris menunjukkan proporsi target di dalam kategori tersebut.


In [ ]:
relationship_categorical_columns = [
    "family_history_with_overweight",
    "FAVC",
    "CAEC",
    "SCC",
    "CALC",
    "MTRANS",
    "SMOKE",
    "Gender",
]

for column in relationship_categorical_columns:
    table = pd.crosstab(
        df_eda[column],
        df_eda["NObeyesdad"],
        normalize="index",
    ).reindex(columns=target_order, fill_value=0)

    print(f"\n=== Proporsi target berdasarkan {column} ===")
    display((table * 100).round(2))

    plt.figure(figsize=(11, 5))
    table.plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title(f"Proporsi Target Berdasarkan {column}")
    plt.xlabel(column)
    plt.ylabel("Proporsi")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="NObeyesdad", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## 13. Korelasi fitur numerik

Korelasi hanya menunjukkan hubungan linier dan tidak membuktikan hubungan sebab-akibat. Matriks ini digunakan sebagai eksplorasi awal, bukan dasar tunggal pemilihan fitur.


In [ ]:
correlation_matrix = df_eda[numeric_columns + ["BMI_EDA"]].corr()
display(correlation_matrix.round(3))


In [ ]:
plt.figure(figsize=(10, 8))
image = plt.imshow(correlation_matrix)
plt.colorbar(image)
plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index,
)
plt.title("Matriks Korelasi Fitur Numerik")
plt.tight_layout()
plt.show()


## 14. Pratinjau target biner sesuai konsep project

Definisi awal:

- `Obesity_Type_I`, `Obesity_Type_II`, dan `Obesity_Type_III` → **Obesity**
- kategori lainnya → **Non-Obesity**

Target ini masih berupa pratinjau EDA. Pembuatan target final dilakukan pada notebook preprocessing agar seluruh keputusan terdokumentasi dengan jelas.


In [ ]:
obesity_classes = {
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
}

df_eda["Obesity_Binary"] = np.where(
    df_eda["NObeyesdad"].isin(obesity_classes),
    "Obesity",
    "Non-Obesity",
)

binary_counts = df_eda["Obesity_Binary"].value_counts()
binary_percentages = (
    df_eda["Obesity_Binary"].value_counts(normalize=True) * 100
).round(2)

binary_summary = pd.DataFrame(
    {
        "count": binary_counts,
        "percentage": binary_percentages,
    }
)

display(binary_summary)

plt.figure(figsize=(7, 4))
binary_counts.plot(kind="bar")
plt.title("Distribusi Target Biner Awal")
plt.xlabel("Kelas")
plt.ylabel("Jumlah")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
binary_numeric_summary = (
    df_eda.groupby("Obesity_Binary", observed=False)[
        ["Age", "FCVC", "NCP", "CH2O", "FAF", "TUE", "Height", "Weight", "BMI_EDA"]
    ]
    .mean()
    .round(3)
)

display(binary_numeric_summary)


In [ ]:
binary_relationship_columns = [
    "FAVC",
    "CAEC",
    "SCC",
    "CALC",
    "MTRANS",
    "family_history_with_overweight",
    "Gender",
    "SMOKE",
]

for column in binary_relationship_columns:
    table = pd.crosstab(
        df_eda[column],
        df_eda["Obesity_Binary"],
        normalize="columns",
    )

    print(f"\n=== Distribusi {column} di dalam setiap kelas biner (%) ===")
    display((table * 100).round(2))


## 15. Fitur kandidat sesuai ruang lingkup NutriSmart AI

Model utama direncanakan berfokus pada pola hidup.

### Kandidat fitur pola makan

- `FAVC`
- `FCVC`
- `NCP`
- `CAEC`
- `CH2O`
- `SCC`
- `CALC`

### Kandidat fitur aktivitas dan perilaku

- `FAF`
- `TUE`
- `MTRANS`
- `SMOKE` sebagai fitur opsional

### Tidak digunakan sebagai fitur utama model pola hidup

- `Height`
- `Weight`
- `BMI_EDA`

### Perlu diuji dalam eksperimen terpisah

- `Age`
- `Gender`
- `family_history_with_overweight`

Tiga variabel terakhir dapat meningkatkan performa, tetapi skor model tidak lagi murni menggambarkan pola hidup. Keputusan final akan ditentukan pada tahap feature selection.


In [ ]:
candidate_lifestyle_features = [
    "FAVC",
    "FCVC",
    "NCP",
    "CAEC",
    "CH2O",
    "SCC",
    "FAF",
    "TUE",
    "CALC",
    "MTRANS",
]

optional_lifestyle_features = ["SMOKE"]

profile_features_for_experiment = [
    "Age",
    "Gender",
    "family_history_with_overweight",
]

excluded_from_lifestyle_model = [
    "Height",
    "Weight",
    "BMI_EDA",
]

feature_plan = pd.DataFrame(
    {
        "group": (
            ["Main lifestyle feature"] * len(candidate_lifestyle_features)
            + ["Optional lifestyle feature"] * len(optional_lifestyle_features)
            + ["Separate experiment"] * len(profile_features_for_experiment)
            + ["Excluded from lifestyle model"] * len(excluded_from_lifestyle_model)
        ),
        "feature": (
            candidate_lifestyle_features
            + optional_lifestyle_features
            + profile_features_for_experiment
            + excluded_from_lifestyle_model
        ),
    }
)

display(feature_plan)


## 16. Ringkasan otomatis hasil EDA

Cell berikut menyusun ringkasan angka utama yang dapat digunakan saat menulis laporan.


In [ ]:
eda_summary = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "missing_values": int(df.isna().sum().sum()),
    "full_duplicates": int(df.duplicated().sum()),
    "numeric_columns": len(numeric_columns),
    "categorical_columns": len(categorical_columns),
    "original_target_classes": int(df["NObeyesdad"].nunique()),
    "binary_obesity_count": int(
        (df_eda["Obesity_Binary"] == "Obesity").sum()
    ),
    "binary_non_obesity_count": int(
        (df_eda["Obesity_Binary"] == "Non-Obesity").sum()
    ),
}

display(pd.Series(eda_summary, name="value").to_frame())


## 17. Kesimpulan EDA

Berdasarkan eksplorasi awal:

1. Dataset memiliki **2.111 baris dan 17 kolom**.
2. Dataset tidak memiliki missing value.
3. Terdapat sejumlah baris duplikat penuh yang harus dihapus pada preprocessing.
4. Target asli terdiri dari tujuh kelas dan distribusinya relatif seimbang.
5. Dataset memiliki kombinasi fitur numerik dan kategorikal.
6. Sejumlah fitur kategorikal memiliki kategori yang sangat jarang.
7. Nilai desimal pada beberapa fitur ordinal berkaitan dengan proses pembentukan data sintetik.
8. Tinggi dan berat badan sangat terkait dengan label target karena membentuk BMI.
9. Memasukkan tinggi, berat, atau BMI ke model akan berisiko membuat model lebih banyak membaca status tubuh daripada pola hidup.
10. Untuk konsep NutriSmart AI, model utama akan diprioritaskan menggunakan pola makan, aktivitas fisik, screen time, dan transportasi.
11. Target biner awal memiliki dua kelas yang cukup seimbang untuk eksperimen klasifikasi.
12. Dataset sesuai untuk final project, tetapi keterbatasan data sintetik dan perbedaan populasi harus dijelaskan dalam laporan.

## Keputusan menuju notebook berikutnya

Pada `02_preprocessing_feature_selection.ipynb`, kita akan:

1. Menghapus duplikat.
2. Membuat target biner final.
3. Memisahkan fitur dan target.
4. Membuat dua skenario fitur:
   - pola hidup saja;
   - pola hidup ditambah profil dasar.
5. Menentukan preprocessing numerik dan kategorikal.
6. Membagi data training dan testing secara stratified.
7. Menyimpan data bersih dan keputusan preprocessing.


## 18. Checklist notebook

Pastikan sebelum melanjutkan:

- [ ] Seluruh cell berhasil dijalankan tanpa error.
- [ ] Dataset terbaca dari `data/raw/`.
- [ ] Jumlah missing value telah diperiksa.
- [ ] Duplikat telah diidentifikasi, tetapi dataset mentah belum diubah.
- [ ] Distribusi target telah dipahami.
- [ ] Hubungan BMI dan target telah dipahami sebagai potensi leakage.
- [ ] Kandidat fitur pola hidup telah disepakati.
- [ ] Kesimpulan EDA telah dicatat untuk laporan.
